# E66b — 추론 컬럼 커버리지 채우기 (689문항)

**런타임 → A100 권장. ③까지 약 40분.**

## 지금 상태

E66의 추론 컬럼은 **효과가 있다** — think 로그비용 RMSE를 처음으로 움직였다:

| 헤드 | 3컬럼 | 4컬럼(+추론) |
|---|---:|---:|
| ax31-light | 0.5556 | 0.5552 |
| ax31 | 0.4580 | 0.4541 |
| **axk1-think** | **0.6768** | **0.6675** (−1.4%) |

34B 컬럼(E59)은 이 숫자를 0.001도 못 움직였다. 그리고 그게 안전계수로 전환된다 —
무초과 premium 계수가 **0.52 → 0.56**, 가중 무초과 점수 0.695710 → 0.696676.

**다만 +0.001로 노이즈 한계 근처다.** 이유는 커버리지: 추론 컬럼이 훈련 행의 **73.9%**만 덮는다.
`pilot.jsonl`(1,951문항)만 라벨링했는데, 이건 `build_pool.py`가 공개 출처에서 되살릴 수 있는 것만
담고 있기 때문이다. E59b에서 34B로 똑같은 문제를 겪었고 `public_all.jsonl`로 0.753 → 0.975를 만들었다.

## 이 노트북이 하는 일

남은 공개 프롬프트 **689개**만 라벨링한다. 이미 끝낸 1,951개는 digest 목록으로 건너뛴다.
남은 것의 구성이 정확히 지금 비어 있는 곳이다:

| family | 개수 |
|---|---:|
| dmmath | 359 |
| longdoc | 114 |
| gsm8k_or_other | 99 |
| truthfulqa | 79 |
| aime | 34 |
| code · ruletaker | 4 |

전부 gold 답이 없는 문항이라 채점은 안 되지만 **상관없다** — 우리가 사는 건 점수가 아니라
**출력 길이**다(`--length-only`로 점수는 어차피 버린다).

In [ ]:
#@title ① 번들 압축 해제
import os, zipfile, glob, shutil
BUNDLE = 'e66b_colab_bundle.zip'
if not os.path.exists(BUNDLE):
    try:
        from google.colab import drive; drive.mount('/content/drive')
        c = glob.glob('/content/drive/MyDrive/**/' + BUNDLE, recursive=True)
        if c: shutil.copyfile(c[0], BUNDLE)
    except Exception as e: print('drive skip:', e)
if not os.path.exists(BUNDLE) or not zipfile.is_zipfile(BUNDLE):
    from google.colab import files; files.upload()
print('zip 정상:', zipfile.is_zipfile(BUNDLE), f'{os.path.getsize(BUNDLE)/1e6:.1f} MB')
with zipfile.ZipFile(BUNDLE) as z: z.extractall('.')
%cd /content/official-router
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title ② 의존성 + 설정 (E66과 동일해야 한다 — 다르면 두 묶음의 길이가 비교 불가해진다)
!pip -q install vllm datasets bitsandbytes 2>&1 | tail -2
!pip -q uninstall -y torchaudio 2>&1 | tail -1
import os, torch, transformers, vllm
GB = torch.cuda.get_device_properties(0).total_memory / 1e9
os.environ.update(
    MODEL='deepseek-ai/DeepSeek-R1-Distill-Qwen-14B',   # E66과 반드시 같은 모델
    N='2', TEMP='0.6', MAXTOK='4096', MAXLEN='16384',   # 같은 샘플 수·온도·길이 한도
    QUANT='' if GB >= 70 else 'bitsandbytes',
    UTIL='0.92', TOKENIZERS_PARALLELISM='false')
print('OK', torch.__version__, f'{GB:.0f} GB', os.environ['MODEL'],
      'quant=' + (os.environ['QUANT'] or 'bf16'))

In [ ]:
#@title ③ 남은 689문항 라벨링 (~40분)
import os
# 이미 커버한 1,951개를 digest 로 제외하고 나머지만 뽑는다
!PYTHONPATH=src python -X utf8 colab-label/build_public_all.py \
    --exclude-digests colab-label/reason_covered_digests.txt \
    --out colab-label/bundle/pool.jsonl
!wc -l colab-label/bundle/pool.jsonl
LAB = 'colab-label/out/labels_pool_T0.6_reason.jsonl'
before = sum(1 for _ in open(LAB, encoding='utf-8')) if os.path.exists(LAB) else 0
print('라벨링 전 행 수:', before)
!PYTHONPATH=src python -X utf8 colab-label/run_labels.py --stage pool \
    --model "$MODEL" --engine vllm --n $N --temp $TEMP \
    --max-model-len $MAXLEN --max-tokens $MAXTOK --gpu-util $UTIL \
    ${QUANT:+--quant $QUANT} --tag _reason \
    --bundle colab-label/bundle --out colab-label/out 2>&1 | grep -v -i warn | tail -25
after = sum(1 for _ in open(LAB, encoding='utf-8'))
print('=== 새로 추가된 행:', after - before, ' (기대: 650~689) ===')
if after == before:
    print('*** 0건이면 위 오류를 그대로 복사해서 알릴 것 ***')

In [ ]:
#@title ④ 커버리지 확인 — 0.97 이상이면 성공
import os, json, hashlib, sys
from pathlib import Path
sys.path.insert(0, 'src')
from ossp_router.heuristic import episode_text
from ossp_router.protocol import load_input

ent = {l.strip() for l in open('colab-label/reason_covered_digests.txt', encoding='utf-8') if l.strip()}
print('E66 시점 커버:', len(ent))
prompts = {}
for line in open('colab-label/bundle/pool.jsonl', encoding='utf-8'):
    if line.strip():
        r = json.loads(line); prompts[r['id']] = r['prompt']
LAB = 'colab-label/out/labels_pool_T0.6_reason.jsonl'
if os.path.exists(LAB):
    for line in open(LAB, encoding='utf-8'):
        if line.strip():
            r = json.loads(line); t = prompts.get(r['id'])
            if t: ent.add(hashlib.sha256(t.encode()).hexdigest())
print('합계 커버:', len(ent))
for split in ('train', 'dev'):
    eps = list(load_input(Path('data/materialized/' + split + '/inputs.json')).episodes)
    hit = sum(hashlib.sha256(episode_text(e).encode()).hexdigest() in ent for e in eps)
    print('  %s 커버리지 %d/%d = %.3f   (E66 시점 0.739)' % (split, hit, len(eps), hit / len(eps)))

In [ ]:
#@title ⑤ 결과 회수 → MyDrive
!cd /content/official-router && zip -qr /content/e66b_out.zip \
    colab-label/out/labels_pool_T0.6_reason.jsonl colab-label/bundle/pool.jsonl
!ls -la /content/e66b_out.zip
import zipfile, os
print('zip 정상:', zipfile.is_zipfile('/content/e66b_out.zip'))
from google.colab import drive; drive.mount('/content/drive')
!cp /content/e66b_out.zip /content/drive/MyDrive/
s, d = '/content/e66b_out.zip', '/content/drive/MyDrive/e66b_out.zip'
print('MyDrive 사본 크기 일치:', os.path.exists(d) and os.path.getsize(d) == os.path.getsize(s))

---

## ⑥ 이후 (지금은 돌리지 말 것)

위까지는 **dev 에서 잴 수 있는** 커버리지를 채운다. 비공개셋 커버리지는 별개이고, 출처에서 렌더링한
풀(`pool.jsonl` 37,882 + `aime.jsonl` 130)을 추론 모델로 돌려야 한다 — **3~5시간**.

④~⑤ 결과로 안전계수가 실제로 올라가는지 먼저 확인하고, 그 다음에 판단하는 게 순서다.
34B 때도 커버리지 완성(E59b)이 dev 기준으로는 EV 중립이었으므로, 이번에도 그럴 수 있다.

In [ ]:
#@title ⑥ (보류) 비공개셋 커버리지용 대량 라벨링 — 3~5시간
# ④~⑤ 결과를 확인하고 지시가 있을 때만 실행할 것.
# !PYTHONPATH=src python -X utf8 colab-label/build_pool.py --out colab-label/bundle --verify
# !cat colab-label/bundle/pilot.jsonl colab-label/bundle/pool.jsonl > colab-label/bundle/all.jsonl
# !PYTHONPATH=src python -X utf8 colab-label/build_pool_ext.py \
#     --have colab-label/bundle/all.jsonl --out colab-label/bundle/ext.jsonl --babilong
# !cat colab-label/bundle/pool.jsonl colab-label/bundle/ext.jsonl > /tmp/p.jsonl && mv /tmp/p.jsonl colab-label/bundle/pool.jsonl
# !PYTHONPATH=src python -X utf8 colab-label/run_labels.py --stage pool \
#     --model "$MODEL" --engine vllm --n $N --temp $TEMP \
#     --max-model-len $MAXLEN --max-tokens $MAXTOK --gpu-util $UTIL \
#     ${QUANT:+--quant $QUANT} --tag _reason \
#     --bundle colab-label/bundle --out colab-label/out 2>&1 | grep -v -i warn | tail -30
print('보류 셀 — ④~⑤ 결과 확인 후 판단')